# Lab 19 — Production-Grade GraphRAG vs Flat RAG

**Submission notebook · AICB-K34 Track 3**

Notebook này là entrypoint thực thi end-to-end cho bài Lab 19. Pipeline được khóa theo policy:

- **FIRST_5000_ROWS_ONLY** — chỉ sử dụng đúng 5.000 dòng đầu của HackerNoon (index 0–4999), giữ nguyên thứ tự, không random sample.
- Golden benchmark canonical: `data/graphrag_golden_50_first5000.csv` (50 câu: factoid, multi-hop, cross-doc).
- Neo4j ingestion dùng `UNWIND` batch + provenance bắt buộc.
- Groq có timeout, pacing, `Retry-After`/exponential backoff và checkpoint để phù hợp free-tier rate limit.
- Bonus: Near-Dedup SimHash/LSH, Community Reports/Global Search, Self-Correction hop2 → hop3 → vector fallback.

> CI đặt `LAB_RUN_MODE=smoke` hoặc `full`. Trên Colab/local có thể đặt biến này trước khi chạy.

In [1]:
import os
from pathlib import Path
import pandas as pd

from lab19_models import DEFAULT_GROQ_MODEL
from lab19_utils import load_golden_dataset, validate_golden_dataset

os.environ.setdefault("GROQ_FAST_MODEL", DEFAULT_GROQ_MODEL)
RUN_MODE = os.getenv("LAB_RUN_MODE", "smoke").strip().lower()
assert RUN_MODE in {"smoke", "full"}

print("Run mode:", RUN_MODE)
print("Source policy: FIRST_5000_ROWS_ONLY")
print("Groq model:", os.environ["GROQ_FAST_MODEL"] )

Run mode: full
Source policy: FIRST_5000_ROWS_ONLY
Groq model: openai/gpt-oss-20b


## 1. Golden dataset validation

Không tạo lại golden answer trong notebook. Dataset do đề/lớp cung cấp được dùng trực tiếp và được validate trước khi benchmark.

In [2]:
GOLDEN_PATH = Path("data/graphrag_golden_50_first5000.csv")
golden_df = load_golden_dataset(GOLDEN_PATH)
golden_report = validate_golden_dataset(golden_df)
display(pd.DataFrame([golden_report]))
display(golden_df.groupby("group").size().rename("questions").to_frame())
assert golden_report["rows"] == 50

,rows,groups,missing_reference_answers,missing_reference_evidence,duplicate_ids
0,50,"[cross-doc, factoid, multi-hop]",0,0,0


,questions
group,
cross-doc,22
factoid,5
multi-hop,23


## 2. Architecture & rubric mapping

`run_lab()` thực hiện tuần tự:

1. Stream **first 5,000 rows** → exact dedup → Near-Dedup SimHash/LSH → chunking.
2. Conservative coreference (chỉ gọi LLM khi chunk có trigger) → NER/RE strict schema.
3. Entity Resolution: manual aliases + FAISS ANN + lexical/type guard + DSU audit.
4. Neo4j constraints/indexes → `UNWIND` bulk node/edge ingestion → zero-missing-provenance assertion.
5. Flat RAG FAISS và Hybrid GraphRAG (entity seed matching + BFS + super-node mitigation).
6. Self-Correction: hop2 → hop3 → vector fallback.
7. Golden benchmark + LLM-as-a-Judge: comprehensiveness, faithfulness, multi-hop reasoning, latency, token usage.
8. Community detection + Community Reports/Global Search.
9. Export rubric evidence và reports.

Các relation của Knowledge Graph vẫn giữ đúng allowlist của đề:
`ACQUIRED, DEVELOPED, INVESTED_IN, FOUNDED, WORKED_AT, PARTNERED_WITH, USES, LEADS`.

In [3]:
from lab19_runtime import run_lab

manifest = run_lab(RUN_MODE)
display(pd.DataFrame([manifest]))

[run] mode=full; source_scope=FIRST_5000_ROWS_ONLY


/opt/hostedtoolcache/Python/3.11.16/x64/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[data] Source policy FIRST_5000_ROWS_ONLY; rows=5000; random_sampling=DISABLED


[data] standardized/exact-dedup=2,119; after-near-dedup=2,114; chunks=2,114


[extract] selected 400 chunks across source rows 0..4997


...

Batches:  50%|█████     | 2/4 [00:00<00:00,  3.43it/s]

Batches:  75%|███████▌  | 3/4 [00:00<00:00,  4.34it/s]

Batches: 100%|██████████| 4/4 [00:00<00:00,  4.85it/s]

[run] completed {"mode": "full", "source_policy": "FIRST_5000_ROWS_ONLY", "source_rows_downloaded": 5000, "articles_after_dedup": 2114, "chunks_indexed": 2114, "chunks_llm_extracted": 400, "triples": 268, "nodes": 418, "golden_questions_evaluated": 50, "entity_audit_rows": 16, "near_dedup_rows": 5, "communities": 164, "invalid_provenance_edges": 0}


,mode,source_policy,source_rows_downloaded,articles_after_dedup,chunks_indexed,chunks_llm_extracted,triples,nodes,golden_questions_evaluated,entity_audit_rows,near_dedup_rows,communities,invalid_provenance_edges
0,full,FIRST_5000_ROWS_ONLY,5000,2114,2114,400,268,418,50,16,5,164,0


## 3. Rubric evidence

Các file dưới đây được sinh từ chính run hiện tại, không hard-code benchmark result.

In [4]:
OUTPUTS = Path("outputs")
expected = [
    "graphrag_eval_results.csv",
    "graphrag_vs_flatrag_summary.csv",
    "entity_resolution_audit.csv",
    "supernode_diagnostics.csv",
    "community_reports.csv",
    "bonus_metrics.csv",
]
for name in expected:
    path = OUTPUTS / name
    print(f"{name}: {'OK' if path.exists() else 'MISSING'}")
    assert path.exists(), path

graphrag_eval_results.csv: OK
graphrag_vs_flatrag_summary.csv: OK
entity_resolution_audit.csv: OK
supernode_diagnostics.csv: OK
community_reports.csv: OK
bonus_metrics.csv: OK


In [5]:
eval_df = pd.read_csv(OUTPUTS / "graphrag_eval_results.csv")
summary_df = pd.read_csv(OUTPUTS / "graphrag_vs_flatrag_summary.csv")
display(summary_df)
display(eval_df[[
    "id", "group",
    "flat_comprehensiveness", "graph_comprehensiveness",
    "flat_faithfulness", "graph_faithfulness",
    "flat_multi_hop_reasoning", "graph_multi_hop_reasoning",
    "flat_latency_s", "graph_latency_s",
    "graph_route", "graph_edges", "graph_supernode_events"
]].head(10))

,question_group,metric,flat_rag,graph_rag,delta_graph_minus_flat
0,cross-doc,Comprehensiveness,3.591,3.955,0.364
1,cross-doc,Faithfulness,3.682,3.955,0.273
2,cross-doc,Multi-hop reasoning,3.455,3.773,0.318
3,cross-doc,Latency (s),2.487,5.140,2.653
4,cross-doc,Token usage,931.091,1797.636,866.545
5,factoid,Comprehensiveness,5.000,5.000,0.000
6,factoid,Faithfulness,5.000,5.000,0.000
7,factoid,Multi-hop reasoning,4.200,5.000,0.800
8,factoid,Latency (s),1.498,4.189,2.691
9,factoid,Token usage,884.600,1496.400,611.800


,id,group,flat_comprehensiveness,graph_comprehensiveness,flat_faithfulness,graph_faithfulness,flat_multi_hop_reasoning,graph_multi_hop_reasoning,flat_latency_s,graph_latency_s,graph_route,graph_edges,graph_supernode_events
0,G5000-01,multi-hop,5,5,5,5,5,5,2.525706,4.879096,hop3+vector,2,0
1,G5000-02,cross-doc,5,5,5,5,5,5,4.039957,4.521348,hop3+vector,1,0
2,G5000-03,factoid,5,5,5,5,5,5,1.290092,4.385255,hop3+vector,2,0
3,G5000-04,cross-doc,5,4,5,4,5,4,2.638627,3.283648,hop2,10,0
4,G5000-05,multi-hop,5,5,5,5,5,5,2.513582,3.097996,hop3+vector,1,0
5,G5000-06,multi-hop,2,3,2,3,2,3,3.154691,8.015540,hop2,13,0
6,G5000-07,cross-doc,5,5,5,5,5,5,2.900618,5.657427,hop2,9,0
7,G5000-08,multi-hop,2,5,2,5,1,5,2.391757,6.361430,hop2,24,0
8,G5000-09,cross-doc,3,2,3,2,2,1,1.646362,1.447569,hop3+vector,0,0
9,G5000-10,multi-hop,3,3,3,3,3,3,4.496873,7.082628,hop2,13,0


## 4. Failure-mode evidence

- **Entity Resolution audit:** `MERGE_MANUAL`, `MERGE_VECTOR`, `REJECT_GUARD`.
- **Super-node:** degree > 100 ⇒ tối đa 50 edge mới nhất, đồng thời có `GLOBAL_EDGE_CAP`.
- **Provenance:** mỗi edge phải có `source_chunk_id`, `published_date`, `evidence`.

In [6]:
audit_df = pd.read_csv(OUTPUTS / "entity_resolution_audit.csv")
supernode_df = pd.read_csv(OUTPUTS / "supernode_diagnostics.csv")
display(audit_df.head(20))
display(supernode_df.head(15))

graph_checks_path = OUTPUTS / "graph_checks.json"
if graph_checks_path.exists():
    import json
    graph_checks = json.loads(graph_checks_path.read_text(encoding="utf-8"))
    display(pd.DataFrame([graph_checks]))
    assert graph_checks["invalid_provenance_edges"] == 0

,type,left,right,similarity,decision
0,Company,MSFT,Microsoft,1.000000,MERGE_MANUAL
1,Company,Fidelity National Information Services Inc.,Fidelity National Information Services,0.924524,MERGE_VECTOR
2,Company,Fidelity National Information Services Inc.,Fidelity National Information Services,1.000000,MERGE_MANUAL
3,Company,Ivy Tech Community College - Columbus,Ivy Tech Community College,0.898756,REJECT_GUARD
4,Company,Broadridge Financial Solutions,Broadridge,0.754731,REJECT_GUARD
5,Company,more than 900 clients,more than 900 clients including more than 75 o...,0.766403,REJECT_GUARD
6,Company,Amazon.com,Amazon,0.847531,REJECT_GUARD
7,Company,AST Spacemobile,SpaceMobile,0.828274,REJECT_GUARD
8,Technology,generative AI,Generative AI Solutions,0.866633,REJECT_GUARD
9,Technology,generative AI,generative AI capabilities,0.856484,REJECT_GUARD


,id,name,type,degree
0,lab19_fb0f4df56fab164ec48722f0,Microsoft,Company,9
1,lab19_1a324b330bd048e7308e53f0,artificial intelligence,Technology,7
2,lab19_cc9c6ee3857729e221d3f6de,ServiceNow,Company,5
3,lab19_0e132222d1315530d6efca52,Google,Company,5
4,lab19_b823fb02e58a12bdc4c38934,NVIDIA,Company,5
5,lab19_e23c5421d4f8b156c0c6101c,UKG,Company,4
6,lab19_773eeb9b7cc008bff365fcdd,OpenAI,Company,4
7,lab19_4f5c599d80e3f459819f8d8f,AI,Technology,3
8,lab19_fa2c6b919de1f8285f8d8d20,WiMi,Company,3
9,lab19_560780a8c7dfa297cfdf94c8,Dragos Partner Program,Technology,3


,nodes,edges,invalid_provenance_edges
0,418,268,0


## 5. Bonus evidence

### Bonus A — Near-Dedup
SimHash64 + LSH buckets; tránh pairwise cosine O(N²), có audit cặp drop/keep.

### Bonus B — Global Search via Community Reports
NetworkX Louvain → ghi `community_id` về Neo4j → tạo community reports → semantic global-search demo.

### Bonus C — Self-Correction Graph Retrieval
GraphRAG bắt đầu hop2; thiếu context thì hop3; vẫn thiếu thì hybrid vector fallback, có stop condition và route diagnostics.

In [7]:
bonus_df = pd.read_csv(OUTPUTS / "bonus_metrics.csv")
community_df = pd.read_csv(OUTPUTS / "community_reports.csv")
display(bonus_df)
display(community_df.head(10))

,bonus,metric,value
0,Near-Dedup SimHash/LSH,near_duplicates_removed,5.0
1,Global Search Community Reports,communities,164.0
2,Self-Correction Graph Retrieval,hop2,36.0
3,Self-Correction Graph Retrieval,hop3,0.0
4,Self-Correction Graph Retrieval,hop3+vector,14.0
5,Global Search Community Reports,llm_summaries,164.0
6,Global Search Community Reports,llm_summary_rate,1.0
7,Low-level / High-level Router,global_routes,2.0
8,Low-level / High-level Router,local_routes,2.0
9,Self-Correction LLM Sufficiency,hop2,3.0


,community_id,size,hubs,dominant_relations,edge_examples,report,summary_method,community_algorithm
0,0,18,artificial intelligence | Google | Meta | data...,DEVELOPED | USES | PARTNERED_WITH | INVESTED_IN,Google -DEVELOPED-> Tensor Processing Unit | G...,This community centers on major technology com...,LLM,networkx.greedy_modularity_communities
1,1,12,Microsoft | AI | tech support | Apple | Linked...,USES | PARTNERED_WITH | DEVELOPED | INVESTED_IN,Microsoft -PARTNERED_WITH-> Dynamics 365 Finan...,This community involves companies like Microso...,LLM,networkx.greedy_modularity_communities
2,2,11,OpenAI | NVIDIA | ServiceNow | generative AI |...,PARTNERED_WITH | DEVELOPED | USES,The Associated Press -PARTNERED_WITH-> OpenAI ...,This community highlights organizations like O...,LLM,networkx.greedy_modularity_communities
3,3,5,UKG | HR payroll and workforce management solu...,PARTNERED_WITH | DEVELOPED,Prolucent Health -PARTNERED_WITH-> UKG | UKG -...,This community focuses on UKG and its partners...,LLM,networkx.greedy_modularity_communities
4,4,5,Smart Eye | AIS Driver Monitoring System | Geo...,DEVELOPED | PARTNERED_WITH,Smart Eye -DEVELOPED-> Driver Monitoring Syste...,Centered on Smart Eye and its development of d...,LLM,networkx.greedy_modularity_communities
5,5,5,Senser | DriveNets | Or Sadeh | Yuval Lev | In...,FOUNDED | WORKED_AT | PARTNERED_WITH,Senser -PARTNERED_WITH-> Intel Ignite | Senser...,"This community includes Senser, DriveNets, and...",LLM,networkx.greedy_modularity_communities
6,6,4,Broadridge Financial Solutions | Syndio | Synd...,PARTNERED_WITH | USES,Syndio -PARTNERED_WITH-> Broadridge Financial ...,This community involves Broadridge Financial S...,LLM,networkx.greedy_modularity_communities
7,7,4,PSG | BioProTTTM FlowSU System | Dover | SumoF...,DEVELOPED | PARTNERED_WITH,PSG -DEVELOPED-> BioProTTTM FlowSU System | PS...,Centered on PSG and its development of special...,LLM,networkx.greedy_modularity_communities
8,8,4,SAM4 | Samotics | thyssenkrupp | technique,PARTNERED_WITH | DEVELOPED | USES,thyssenkrupp -PARTNERED_WITH-> Samotics | Samo...,"This community includes thyssenkrupp, Samotics...",LLM,networkx.greedy_modularity_communities
9,9,4,Aran Azarzar | JFrog | Shlomi Ben Haim | JFrog...,LEADS | FOUNDED,Aran Azarzar -LEADS-> JFrog | Aran Azarzar -LE...,This community features individuals Aran Azarz...,LLM,networkx.greedy_modularity_communities


## 6. Submission artifacts

Sau full run, repo/artifact cần có:

- executed notebook này;
- `outputs/graphrag_eval_results.csv`;
- `outputs/graphrag_vs_flatrag_summary.csv`;
- audit/diagnostic/bonus CSV;
- `reports/lab_report.md`;
- `reports/technical_defense.md`;
- `reports/failure_analysis.md`;
- `reports/reflection_ChuNguyenTuanAnh.md`.

Không hard-code API key/password trong notebook hoặc repo.